In [15]:
# Midnight issues only: explicitly separate from the failed six-hourly attempt.
seasonal_source="\"\"\"Run only in Colab: three prespecified historical seasons, no test-set tuning.\"\"\"\nimport argparse, hashlib, json, pathlib, time\nimport joblib, numpy as np, pandas as pd\nfrom ml import train\nfrom ml.features import FEATURES\nfrom ml.predict import probability\n\np=argparse.ArgumentParser();p.add_argument('--version',required=True);a=p.parse_args()\nroot=pathlib.Path.cwd();out=pathlib.Path('/content/fra-seasonal24-results')/a.version;out.mkdir(parents=True,exist_ok=True)\nstarted=time.time(); frame,truth,obs,skipped=train.dataset(24)\nkeys=['issue_time','time','lead_h']\nmanifest={'version':a.version,'issue_stride_h':24,'months':['2026-01','2026-04','2026-07'],'feature_rows':len(frame),'skipped':skipped,'runs':{},'source_sha256':{str(f.relative_to(root)):hashlib.sha256(f.read_bytes()).hexdigest() for f in (root/'ml').rglob('*.py')},'data_sha256':{name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in ['data/ground_truth/dfld_hourly.csv','data/eddf_metar.parquet','data/openmeteo_previous_runs.parquet','data/openmeteo_previous_runs_multimodel.parquet','data/openmeteo_previous_runs_pressure.parquet']}}\nfor month in manifest['months']:\n    period=pd.Period(month,freq='M');start=period.start_time.tz_localize('UTC');end=(period+1).start_time.tz_localize('UTC')\n    fit=frame[frame.label_available_at<start].copy()\n    test=frame[(frame.issue_time>=start)&(frame.issue_time<end)&(frame.label_available_at<end)].copy()\n    assert len(fit)>0 and len(test)>0 and fit.label_available_at.max()<test.issue_time.min()\n    assert fit.y_od.equals(train.without_stuck_feed(fit).y_od),'Training label policies differ'\n    assert test.lead_h.between(1,48).all()\n    assert not test.duplicated(keys).any()\n    print('FITTING',a.version,month,len(fit),len(test),flush=True)\n    model=train.fit(fit);pred=train.predict(model,test)\n    pred['p_18_usable_raw']=probability(model['r18'],test[FEATURES])\n    pred['certainty']=100*np.minimum(abs(2*pred.p_od25-1),abs(2*pred.p_18_usable-1))\n    for col in ['p_od25','p_od25_raw','p_18_usable','p_18_usable_raw']:\n        assert np.isfinite(pred[col]).all() and pred[col].between(0,1).all()\n    cols=keys+['y_od','y_r18','od_at_issue','persistence_18','rule_od25','rule_18_usable','rule_od25_ens','rule_18_ens','changed_within_lead','p_od25','p_od25_raw','p_18_usable','p_18_usable_raw','certainty']\n    pred[cols].to_parquet(out/f'{month}-predictions.parquet',index=False)\n    tk=fit[keys+['y_od','y_r18','label_available_at']].sort_values(keys).to_csv(index=False)\n    (out/f'{month}-training-keys.csv').write_text(tk)\n    joblib.dump(model,out/f'{month}-model.joblib',compress=3)\n    manifest['runs'][month]={'train_rows':len(fit),'test_rows':len(test),'train_label_end':fit.label_available_at.max().isoformat(),'training_keys_sha256':hashlib.sha256(tk.encode()).hexdigest(),'model_sha256':hashlib.sha256((out/f'{month}-model.joblib').read_bytes()).hexdigest()}\n    print('SEASON_FIT_COMPLETE',a.version,month,flush=True)\nmanifest['sampling_note']='Deadline-limited midnight-issue seasonal check; not directly comparable with six-hourly team folds.'\nmanifest['elapsed_s']=time.time()-started\n(out/'manifest.json').write_text(json.dumps(manifest,indent=2,default=str))\nprint('SEASONAL_VERSION_COMPLETE',a.version,round(time.time()-started,1),flush=True)\n"
seasonal24_out=pathlib.Path('/content/fra-seasonal24-results');seasonal24_out.mkdir(exist_ok=True)
active=[]
for version,folder in [('v5','/content/fra-v5-icon'),('v6','/content/fra-v6-verify')]:
    root=pathlib.Path(folder);(root/'seasonal_fit_24h.py').write_text(seasonal_source)
    env=os.environ.copy();env['OMP_NUM_THREADS']='1';env['OPENBLAS_NUM_THREADS']='1'
    child=subprocess.Popen([sys.executable,'-u','seasonal_fit_24h.py','--version',version],cwd=root,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    active.append((version,child))
for version,child in active:
    with (seasonal24_out/f'{version}-run.log').open('w') as log:
        for line in child.stdout:
            print(version,line,end='',flush=True);log.write(line);log.flush()
    status=child.wait();print('SEASONAL24_RETURN_CODE',version,status,flush=True)
    assert status==0
(V6ROOT/'seasonal24_audit.py').write_text("\"\"\"Common v6 scorer for both versions; no fitting or model selection.\"\"\"\nimport json,pathlib\nimport numpy as np,pandas as pd\nfrom ml import quality\nROOT=pathlib.Path('/content/fra-v6-verify'); OUT=pathlib.Path('/content/fra-seasonal24-results')\nhalf=pd.read_csv(ROOT/'data/ground_truth/dfld_half_hourly.csv.gz',dtype={'od':str})\nlocal=pd.to_datetime(half.date_local)+pd.to_timedelta(half.hour_local,unit='h')+pd.to_timedelta(half.minute_local,unit='m')\nhalf['time']=local.dt.tz_localize('Europe/Berlin',ambiguous='NaT',nonexistent='NaT').dt.tz_convert('UTC');half=half.dropna(subset=['time'])\nms={v:json.loads((OUT/v/'manifest.json').read_text()) for v in ('v5','v6')}\nassert ms['v5']['data_sha256']==ms['v6']['data_sha256']\nrecords=[];events={};parity=[];checks=[]\nreference=json.loads((ROOT/'reference-quality.json').read_text()); keys=['issue_time','time','lead_h']\nfor month in ['2026-01','2026-04','2026-07']:\n    frames={v:pd.read_parquet(OUT/v/f'{month}-predictions.parquet').sort_values(keys).reset_index(drop=True) for v in ('v5','v6')}\n    assert ms['v5']['runs'][month]['training_keys_sha256']==ms['v6']['runs'][month]['training_keys_sha256'],'Training keys/labels differ'\n    pd.testing.assert_frame_equal(frames['v5'][keys+['y_od','y_r18']],frames['v6'][keys+['y_od','y_r18']])\n    events[month]={}\n    for target,y,p,r,b in [('od','y_od','p_od25','rule_od25','od_at_issue'),('r18','y_r18','p_18_usable','rule_18_usable','persistence_18')]:\n        eligible=np.ones(len(frames['v5']),dtype=bool)\n        for v in ('v5','v6'): eligible &= frames[v][[y,p,r,b]].notna().all(axis=1).to_numpy()\n        for version in ('v5','v6'):\n            frame=frames[version].loc[eligible].copy()\n            for row in quality.skill(frame,target):\n                sub=frame[frame.lead_h.eq(row['lead_h'])]\n                assert row['n']==len(sub)\n                assert np.isclose(((sub[p]-sub[y])**2).mean(),row['model_brier'],rtol=0,atol=1e-12)\n                assert np.isclose(((sub[p]>=.5)==sub[y]).mean(),row['model_acc'],rtol=0,atol=1e-12)\n                for method,prefix in [('blend','model'),('raw','raw_model'),('rules','rule'),('persistence','persistence')]:\n                    records.append(dict(month=month,version=version,target=target,method=method,lead_h=row['lead_h'],n=row['n'],accuracy=row[prefix+'_acc'],brier=row[prefix+'_brier'],changed_n=row['change_n'],changed_accuracy=row[prefix+'_change_acc']))\n        ref=next(f for f in reference['folds'] if f['month']==month)['out_of_sample'][target]\n        for actual,expected in (zip(quality.skill(frames['v6'],target),ref) if ms['v6'].get('issue_stride_h',6)==6 else []):\n            for field in ('model_acc','model_brier','raw_model_acc','raw_model_brier'):\n                parity.append(dict(month=month,target=target,lead_h=actual['lead_h'],field=field,n_match=actual['n']==expected['n'],actual=actual[field],reference=expected[field],delta=actual[field]-expected[field],within_1e_9=abs(actual[field]-expected[field])<1e-9))\n    for version in ('v5','v6'):\n        f=frames[version]\n        events[month][version]={'next_change':quality.next_change_timing(f,half),'unique_changes':quality.od_change_events(f,half),'r18':quality.r18_events(f)}\n        f.to_csv(OUT/version/f'{month}-prediction-traces.csv',index=False)\n    checks.append({'month':month,'test_rows':len(frames['v5']),'training_keys_and_labels_identical':True,'test_keys_and_labels_identical':True})\n    print('MATCHED_SEASON_AUDIT_PASS',month,len(frames['v5']),flush=True)\ntable=pd.DataFrame(records);table.to_csv(OUT/'seasonal_metrics.csv',index=False)\npd.DataFrame(parity).to_csv(OUT/'reference_parity.csv',index=False)\n(OUT/'event_metrics.json').write_text(json.dumps(events,indent=2,default=str))\nsummary={'months':['2026-01','2026-04','2026-07'],'checks':checks,'same_raw_data_hashes':True,'v6_reference_parity_all':(all(r['within_1e_9'] and r['n_match'] for r in parity) if parity else None),'issue_stride_h':24,'metric_rows':len(table),'scorer':'v6 quality.py for both versions; complete windows','limitations':['Reused historical development months; not untouched forward evaluation.','No tuning or seasonal model selection from these results.','R18 labels are weather-rule proxies.','Version-specific feature construction differs; raw data and train/test keys and labels match.','Lead rows and 48-hour windows overlap; no independent-row significance claim.']}\n(OUT/'audit.json').write_text(json.dumps(summary,indent=2))\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nplt.rcParams['svg.fonttype']='none'\nfig,axes=plt.subplots(1,3,figsize=(12,4),sharey=True)\nfor ax,month in zip(axes,summary['months']):\n    for version in ('v5','v6'):\n        s=table[(table.month==month)&(table.version==version)&(table.target=='od')&(table.method=='blend')]\n        ax.plot(s.lead_h,s.brier,'o-',label=version)\n    ax.set_title(month);ax.set_xticks([1,6,24,48]);ax.set_xlabel('Lead (hours)');ax.grid(alpha=.2)\naxes[0].set_ylabel('OD Brier (lower is better)');axes[-1].legend();fig.suptitle('Fresh seasonal v5 / v6 comparison - matched rows; historical development data')\nfig.tight_layout();fig.savefig(OUT/'seasonal_comparison.svg');fig.savefig(OUT/'seasonal_comparison.png',dpi=150)\nprint('SEASONAL_OD_RESULTS',table[(table.target=='od')&(table.method=='blend')].to_csv(index=False),flush=True)\nprint('SEASONAL_REFERENCE_PARITY',summary['v6_reference_parity_all'],flush=True)\nprint('SEASONAL_AUDIT_COMPLETE',flush=True)\n")
proc=subprocess.run([sys.executable,'-u','seasonal24_audit.py'],cwd=V6ROOT,text=True,capture_output=True)
(seasonal24_out/'audit.log').write_text(proc.stdout+proc.stderr)
print(proc.stdout,proc.stderr,flush=True)
assert proc.returncode==0
exported={}
for label,folder in [('october',V6OUT),('seasonal24',seasonal24_out)]:
    for path in folder.rglob('*'):
        if path.is_file() and path.suffix in ('.json','.csv','.svg','.log','.txt') and 'training' not in path.name and 'prediction' not in path.name:
            exported[label+'/'+str(path.relative_to(folder))]=path.read_text()
print('V6_EXPORT_BEGIN')
print(json.dumps(exported))
print('V6_EXPORT_END')


v5 Flagged 103 recorded hours as a stuck feed
v5 Features 500/1030 issue times
v5 Features 1000/1030 issue times
v5 Dataset 48,576 rows, 1,012 issues; 18 skipped
v5 FITTING v5 2026-01 34150 1414
v5 SEASON_FIT_COMPLETE v5 2026-01
v5 FITTING v5 2026-04 38472 1368
v5 SEASON_FIT_COMPLETE v5 2026-04
v5 FITTING v5 2026-07 42840 1416
v5 SEASON_FIT_COMPLETE v5 2026-07
v5 SEASONAL_VERSION_COMPLETE v5 115.0
SEASONAL24_RETURN_CODE v5 0
v6 Flagged 103 recorded hours for rule-disagreement sensitivity
v6 Features 500/1030 issue times
v6 Features 1000/1030 issue times
v6 Dataset 48,576 rows, 1,012 issues; 18 skipped
v6 FITTING v6 2026-01 34150 1414
v6 SEASON_FIT_COMPLETE v6 2026-01
v6 FITTING v6 2026-04 38472 1368
v6 SEASON_FIT_COMPLETE v6 2026-04
v6 FITTING v6 2026-07 42840 1416
v6 SEASON_FIT_COMPLETE v6 2026-07
v6 SEASONAL_VERSION_COMPLETE v6 148.9
SEASONAL24_RETURN_CODE v6 0
MATCHED_SEASON_AUDIT_PASS 2026-01 1414
MATCHED_SEASON_AUDIT_PASS 2026-04 1368
MATCHED_SEASON_AUDIT_PASS 2026-07 1416
SEASONA